# Natural Language Understanding

Ziele der heutigen Übung:
- Erstellung eines Datensatzes an Nutzeräußerungen mit Hilfe eines LLMs
- Intent Klassifikation regelbasiert
- Implementierung einer NLU auf Basis der Library stanza

## Erstellung Datensatz

Wir möchten in der Übung einen Datensatz nutzen, welcher Nutzeräußerungen zu zwei Intents enthält:
1) ```Pizza_Bestellen```: Ein Nutzer soll mit einem Sprachdialogsystem eine Bestellung für Pizza aufgeben können
2) ```Oeffnungszeiten_Erfragen```: Ein Nutzer soll mit einem Sprachdialogsystem erfragen können, ob die Pizzeria geöffnet hat.

Weiterhin sollen zu jedem Intent folgende Entities möglich sein:
1) ```Pizza_Bestellen```: Belag, Größe, Extra_Käse
2) ```Oeffnungszeiten_Erfragen```: Wochentag

Es geht nichts über reale Dialoge, aber zur ersten Implmenetierung hilft uns ein LLM ungemein, indem es mit einem einfachen Prompt einen entsprechenden synthetischen Datensatz generieren kann.

Gehe mit einem LLM deiner Wahl folgendermaßen vor:
1) Definiere zuerst die Persona des Sprechers, d.h. wer soll mit unserem Sprachdialogsystem interagieren. 
2) Erstelle einen Prompt mit der Definition des Sprechers sowie mit der Spezifikation für Intents/Entities (s.o.)
3) Füge das gewünschte Ausgabeformat dem Prompt hinzu: 200 Nutzeräußerungen als .CSV-Datei mit folgenden Spalten: "utterance" als String, "intent" als String, "entities" als JSON


Wichtige Einschränkung: der Einfachheit halber betrachten wir hier nur den Happy-Path, d.h. wir haben keine Äußerungen die zu keinem der beiden Intents gehören und keine Äußerungen mit Fehlern.


In [231]:
import pandas as pd
pd.set_option('display.max_colwidth', 400)

CSV_FILE = "pizza_utterances.csv"
df = pd.read_csv(CSV_FILE, sep=";", skiprows=1)
df.head()

,utterance,intent,entities
0,Hat das Geschäft am Freitag geöffnet?,Oeffnungszeiten_Erfragen,"{""Wochentag"": ""Freitag""}"
1,Bin ich am Mittwoch abends noch bei Ihnen richtig?,Oeffnungszeiten_Erfragen,"{""Wochentag"": ""Mittwoch""}"
2,"Bitte eine Thunfisch-Pizza, Größe mittlere, ohne extra Käse.",Pizza_Bestellen,"{""Belag"": ""Thunfisch"", ""Groesse"": ""mittlere"", ""Extra_Kaese"": ""nein""}"
3,"Bitte eine Vegetarische, Größe große, ohne extra Käse.",Pizza_Bestellen,"{""Belag"": ""Vegetarische"", ""Groesse"": ""große"", ""Extra_Kaese"": ""nein""}"
4,Bis wann haben Sie Samstag offen?,Oeffnungszeiten_Erfragen,"{""Wochentag"": ""Samstag""}"


Um im folgenden einfacher mit dem Datensatz arbeiten zu können, erstellen wir für jeden Intent einen extra dataframe:

In [232]:
df_order = df[df['intent'] == 'Pizza_Bestellen']
df_opening_hours = df[df['intent'] == 'Oeffnungszeiten_Erfragen']                                                               

Anschließend teilen wir die Daten in einen 20:80 train-test Split (! genau andersrum als beim klassischen Machine Learning!). Wir schauen uns die 20 Trainingsdaten an und versuchen daraus unsere Regeln für die NLU abzuleiten. Im Testset testen wir, wie gut diese funktionieren.

In [233]:
df_order_train = df_order[:19]
df_order_test = df_order[20:]
df_opening_hours_train = df_opening_hours[:19]
df_opening_hours_test = df_opening_hours[20:]

In [234]:
df_order_train

,utterance,intent,entities
2,"Bitte eine Thunfisch-Pizza, Größe mittlere, ohne extra Käse.",Pizza_Bestellen,"{""Belag"": ""Thunfisch"", ""Groesse"": ""mittlere"", ""Extra_Kaese"": ""nein""}"
3,"Bitte eine Vegetarische, Größe große, ohne extra Käse.",Pizza_Bestellen,"{""Belag"": ""Vegetarische"", ""Groesse"": ""große"", ""Extra_Kaese"": ""nein""}"
5,Eine Spinat-Pizza in Größe 32 cm bitte mit extra Käse obendrauf.,Pizza_Bestellen,"{""Belag"": ""Spinat"", ""Groesse"": ""32 cm"", ""Extra_Kaese"": ""ja""}"
6,"Bitte eine Salami, Größe große, ohne extra Käse.",Pizza_Bestellen,"{""Belag"": ""Salami"", ""Groesse"": ""große"", ""Extra_Kaese"": ""nein""}"
8,Gib mir bitte eine große Margherita-Pizza mit extra Käse.,Pizza_Bestellen,"{""Belag"": ""Margherita"", ""Groesse"": ""große"", ""Extra_Kaese"": ""ja""}"
10,Eine Hawaii Pizza bitte mit doppeltem Käse.,Pizza_Bestellen,"{""Belag"": ""Hawaii"", ""Extra_Kaese"": ""ja""}"
12,"Ich hätte gerne eine Pizza Spinat, am liebsten extra große mit zusätzlichem Käse.",Pizza_Bestellen,"{""Belag"": ""Spinat"", ""Groesse"": ""extra große"", ""Extra_Kaese"": ""ja""}"
13,Ich nehme die 32 cm Quattro Formaggi mit doppeltem Käse.,Pizza_Bestellen,"{""Belag"": ""Quattro Formaggi"", ""Groesse"": ""32 cm"", ""Extra_Kaese"": ""ja""}"
14,Ich möchte eine 28 cm Pizza Quattro Formaggi bestellen mit doppeltem Käse.,Pizza_Bestellen,"{""Belag"": ""Quattro Formaggi"", ""Groesse"": ""28 cm"", ""Extra_Kaese"": ""ja""}"
15,Können Sie mir eine mittlere Pizza Peperoni machen mit extra Käse?,Pizza_Bestellen,"{""Belag"": ""Peperoni"", ""Groesse"": ""mittlere"", ""Extra_Kaese"": ""ja""}"


In [235]:
df_opening_hours_train

,utterance,intent,entities
0,Hat das Geschäft am Freitag geöffnet?,Oeffnungszeiten_Erfragen,"{""Wochentag"": ""Freitag""}"
1,Bin ich am Mittwoch abends noch bei Ihnen richtig?,Oeffnungszeiten_Erfragen,"{""Wochentag"": ""Mittwoch""}"
4,Bis wann haben Sie Samstag offen?,Oeffnungszeiten_Erfragen,"{""Wochentag"": ""Samstag""}"
7,Bin ich am Samstag abends noch bei Ihnen richtig?,Oeffnungszeiten_Erfragen,"{""Wochentag"": ""Samstag""}"
9,Haben Sie am Sonntag geöffnet?,Oeffnungszeiten_Erfragen,"{""Wochentag"": ""Sonntag""}"
11,Hat das Geschäft am Donnerstag geöffnet?,Oeffnungszeiten_Erfragen,"{""Wochentag"": ""Donnerstag""}"
16,Bin ich am Sonntag abends noch bei Ihnen richtig?,Oeffnungszeiten_Erfragen,"{""Wochentag"": ""Sonntag""}"
17,Was sind Ihre Öffnungszeiten am Freitag?,Oeffnungszeiten_Erfragen,"{""Wochentag"": ""Freitag""}"
20,Haben Sie am Samstag geöffnet?,Oeffnungszeiten_Erfragen,"{""Wochentag"": ""Samstag""}"
21,Sind Sie am Donnerstag auch vormittags offen?,Oeffnungszeiten_Erfragen,"{""Wochentag"": ""Donnerstag""}"


## NLU mit einfachen Regeln

In [236]:
# Regular expression may help us here, however, it also works with string comparision
import re

Wir schauen uns das Trainingsdatenset genau an und versuchen Wörter zu identifizieren, die auf den einen oder anderen Intent hinweisen:

In [237]:
df_order_train['utterance']

2                                           Bitte eine Thunfisch-Pizza, Größe mittlere, ohne extra Käse.
3                                                 Bitte eine Vegetarische, Größe große, ohne extra Käse.
5                                       Eine Spinat-Pizza in Größe 32 cm bitte mit extra Käse obendrauf.
6                                                       Bitte eine Salami, Größe große, ohne extra Käse.
8                                              Gib mir bitte eine große Margherita-Pizza mit extra Käse.
10                                                           Eine Hawaii Pizza bitte mit doppeltem Käse.
12                     Ich hätte gerne eine Pizza Spinat, am liebsten extra große mit zusätzlichem Käse.
13                                              Ich nehme die 32 cm Quattro Formaggi mit doppeltem Käse.
14                            Ich möchte eine 28 cm Pizza Quattro Formaggi bestellen mit doppeltem Käse.
15                                    Können Sie mir ei

Im Folgenden erstellen wir eine Maske, welche für jede Zeile in unserem Test-Datensatz den Intent abhängig unserer identifizierten Wörter für ein Intent findet. Gebe danach aus wie viele Äußerungen korrekt klassifiziert wurden:

In [238]:
intent_keywords_order = ['bestellen', 'pizza']

regex = '|'.join(intent_keywords_order)
mask = df_order_test['utterance'].str.contains(regex, na=False, flags=re.IGNORECASE, regex=True)

print(f"Korrekt klassifizierte Intents: {int(mask.value_counts().values[0])} von {len(df_order_test)}")

Korrekt klassifizierte Intents: 64 von 80


Dasselbe können wir nun noch für das Intent ```Oeffnungszeiten_Erfragen```machen:

In [246]:
df_opening_hours_train['utterance']

0                             Hat das Geschäft am Freitag geöffnet?
1                Bin ich am Mittwoch abends noch bei Ihnen richtig?
4                                 Bis wann haben Sie Samstag offen?
7                 Bin ich am Samstag abends noch bei Ihnen richtig?
9                                    Haben Sie am Sonntag geöffnet?
11                         Hat das Geschäft am Donnerstag geöffnet?
16                Bin ich am Sonntag abends noch bei Ihnen richtig?
17                         Was sind Ihre Öffnungszeiten am Freitag?
20                                   Haben Sie am Samstag geöffnet?
21                    Sind Sie am Donnerstag auch vormittags offen?
22                    Kann ich am Dienstag noch eine Pizza abholen?
23      Liefern Sie auch, wenn ich Donnerstag spät abends bestelle?
26                             Bis wann haben Sie Donnerstag offen?
27                      Von wann bis wann haben Sie am Freitag auf?
29                         Was sind Ihre Öffnung

In [239]:
intent_keywords_opening_hours = ['Öffnungszeiten', 'geöffnet', 'offen']

regex = '|'.join(intent_keywords_opening_hours)
mask = df_opening_hours_test['utterance'].str.contains(regex, na=False, flags=re.IGNORECASE, regex=True)

print(f"Korrekt klassifizierte Intents: {int(mask.value_counts().values[0])} von {len(df_order_test)}")

Korrekt klassifizierte Intents: 43 von 80


Brainstorme mit deinem Nachbarn, wie man die Erkennung von Entities implementieren könnte. Gehe auf Entities ein, die nicht eindeutig mit einem Wort abbildbar sind, wie zum Beispiel:
```Ich bestelle eine große Pizza Salami mit Käse``` vs. ```Ich bestelle eine extra große Pizza Salami ohne Käse```

## NLU mit Stanza

Bessere Ergebnisse erzielen wir, wenn wir die Äußerung mit NLP-Verfahren analysieren lassen und aus diesen Ergebnissen versuchen unsere Intents und Entities zu identifizieren. Wir möchten hierfür die Bibliothek https://stanfordnlp.github.io/stanza/ einsetzen. Installiere sie mit pip:

In [ ]:
pip install stanza

Anschließend initialisieren wir die Pipeline mit folgenden ```processors='tokenize, pos, lemma, depparse, ner'```. Zusätzlich benötigt sie noch die Sprache in welcher unsere Nutzeräußerungen sind.

Anschließend lassen wir die beiden Äußerungen ```Ich bestelle eine große Pizza Salami mit Käse``` und ```Ich bestelle eine extra große Pizza Salami ohne Käse``` durch die Pipeline analysieren und ausgeben.

Speziell hilfreich für die Interpretation sind folgende Werte:
- lemma
- id und head
- upos oder xpos

Eine Erklärung dazu findest du in der Doku: https://stanfordnlp.github.io/stanza/data_objects.html 

In [241]:
import stanza

nlp = stanza.Pipeline(lang='de', processors='tokenize, pos, lemma, depparse, ner')

order_with_cheese = nlp('Ich bestelle eine große Pizza Salami mit Käse').to_dict()
order_without_cheese = nlp('Ich bestelle eine große Pizza Salami ohne Käse').to_dict()

display(order_with_cheese)
display(order_without_cheese)

2026-10-22 13:26:12 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-10-22 13:26:13 INFO: Downloaded file to /Users/winkelmannsv/Library/Caches/stanza/1.15.0/resources/resources.json
2026-10-22 13:26:13 WARNING: Language de package default expects mwt, which has been added
2026-10-22 13:26:13 INFO: Loading these models for language: de (German):
| Processor | Package             |
-----------------------------------
| tokenize  | combined_nocharlm   |
| mwt       | combined            |
| pos       | combined_charlm     |
| lemma     | combined_nocharlm   |
| depparse  | combined_charlm     |
| ner       | germeval2014_charlm |

2026-10-22 13:26:13 INFO: Using device: cpu
2026-10-22 13:26:13 INFO: Loading: tokenize
2026-10-22 13:26:13 INFO: Loading: mwt
2026-10-22 13:26:13 INFO: Loading: pos
2026-10-22 13:26:14 INFO: Loading: lemma
2026-1

[[{'id': 1,
   'text': 'Ich',
   'lemma': 'ich',
   'upos': 'PRON',
   'xpos': 'PPER',
   'feats': 'Case=Nom|Number=Sing|Person=1|PronType=Prs',
   'head': 2,
   'deprel': 'nsubj',
   'start_char': 0,
   'end_char': 3,
   'ner': 'O',
   'multi_ner': ('O',)},
  {'id': 2,
   'text': 'bestelle',
   'lemma': 'bestellen',
   'upos': 'VERB',
   'xpos': 'VVFIN',
   'feats': 'Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbForm=Fin',
   'head': 0,
   'deprel': 'root',
   'start_char': 4,
   'end_char': 12,
   'ner': 'O',
   'multi_ner': ('O',)},
  {'id': 3,
   'text': 'eine',
   'lemma': 'ein',
   'upos': 'DET',
   'xpos': 'ART',
   'feats': 'Case=Acc|Definite=Ind|Gender=Fem|NumType=Card|Number=Sing|PronType=Art',
   'head': 5,
   'deprel': 'det',
   'start_char': 13,
   'end_char': 17,
   'ner': 'O',
   'multi_ner': ('O',)},
  {'id': 4,
   'text': 'große',
   'lemma': 'groß',
   'upos': 'ADJ',
   'xpos': 'ADJA',
   'feats': 'Case=Acc|Degree=Pos|Gender=Fem|Number=Sing',
   'head': 5,
   'deprel':

[[{'id': 1,
   'text': 'Ich',
   'lemma': 'ich',
   'upos': 'PRON',
   'xpos': 'PPER',
   'feats': 'Case=Nom|Number=Sing|Person=1|PronType=Prs',
   'head': 2,
   'deprel': 'nsubj',
   'start_char': 0,
   'end_char': 3,
   'ner': 'O',
   'multi_ner': ('O',)},
  {'id': 2,
   'text': 'bestelle',
   'lemma': 'bestellen',
   'upos': 'VERB',
   'xpos': 'VVFIN',
   'feats': 'Mood=Ind|Number=Sing|Person=1|Tense=Pres|VerbForm=Fin',
   'head': 0,
   'deprel': 'root',
   'start_char': 4,
   'end_char': 12,
   'ner': 'O',
   'multi_ner': ('O',)},
  {'id': 3,
   'text': 'eine',
   'lemma': 'ein',
   'upos': 'DET',
   'xpos': 'ART',
   'feats': 'Case=Acc|Definite=Ind|Gender=Fem|NumType=Card|Number=Sing|PronType=Art',
   'head': 5,
   'deprel': 'det',
   'start_char': 13,
   'end_char': 17,
   'ner': 'O',
   'multi_ner': ('O',)},
  {'id': 4,
   'text': 'große',
   'lemma': 'groß',
   'upos': 'ADJ',
   'xpos': 'ADJA',
   'feats': 'Case=Acc|Degree=Pos|Gender=Fem|Number=Sing',
   'head': 5,
   'deprel':

Nun machen wir das für alle Äußerungen und speichern die Stanza Ergebnisse in die Dataframes:

In [242]:
df_opening_hours_train['nlu'] = df_opening_hours_train.apply(lambda row: nlp(row['utterance']).to_dict(), axis=1)
df_opening_hours_test['nlu'] = df_opening_hours_test.apply(lambda row: nlp(row['utterance']).to_dict(), axis=1)
df_order_train['nlu'] = df_order_train.apply(lambda row: nlp(row['utterance']).to_dict(), axis=1)
df_order_test['nlu'] = df_order_test.apply(lambda row: nlp(row['utterance']).to_dict(), axis=1)

Einzelne NLU-Ergebnisse können wir uns folgendermaßen anschauen:

In [243]:
index = 2

display(df_order_train['nlu'].loc[[index]].values[0])

[[{'id': 1,
   'text': 'Bitte',
   'lemma': 'Bitte',
   'upos': 'INTJ',
   'xpos': 'ITJ',
   'head': 0,
   'deprel': 'root',
   'start_char': 0,
   'end_char': 5,
   'ner': 'O',
   'multi_ner': ('O',)},
  {'id': 2,
   'text': 'eine',
   'lemma': 'ein',
   'upos': 'DET',
   'xpos': 'ART',
   'feats': 'Case=Nom|Definite=Ind|Gender=Fem|NumType=Card|Number=Sing|PronType=Art',
   'head': 5,
   'deprel': 'det',
   'start_char': 6,
   'end_char': 10,
   'ner': 'O',
   'multi_ner': ('O',)},
  {'id': 3,
   'text': 'Thunfisch',
   'lemma': 'Thunfisch',
   'upos': 'NOUN',
   'xpos': 'NN',
   'feats': 'Case=Nom|Gender=Fem|Number=Sing',
   'head': 5,
   'deprel': 'compound',
   'start_char': 11,
   'end_char': 20,
   'ner': 'O',
   'multi_ner': ('O',),
   'misc': 'SpaceAfter=No'},
  {'id': 4,
   'text': '-',
   'lemma': '-',
   'upos': 'PUNCT',
   'xpos': '$(',
   'head': 3,
   'deprel': 'punct',
   'start_char': 20,
   'end_char': 21,
   'ner': 'O',
   'multi_ner': ('O',),
   'misc': 'SpaceAfter=N

Nehme dir nun verschiedene Sätze aus dem Trainingsdatensatz und baue eine eigene NLU, die Intents und Entities erkennen kann: 

In [ ]:
def nlu(stanza_result: dict) -> tuple[str, dict]:
    ret_intent = None
    ret_entities = {}
    
    intent_lemmas_opening_hours = ['öffnen', 'offen', 'abholen', 'Öffnungszeit']
    intent_lemmas_order = ['bestellen', 'pizza']
    weekdays = ['montag', 'dienstag', 'mittwoch', 'donnerstag', 'freitag', 'samstag', 'sonntag']

    for word in stanza_result[0]:
        lemma_or_text = word['lemma'] if 'lemma' in word else word['text']
        if lemma_or_text in intent_lemmas_order:
            ret_intent = 'Pizza_Bestellen'
        if lemma_or_text in intent_lemmas_opening_hours:
            ret_intent = 'Oeffnungszeiten_Erfragen'
        if lemma_or_text.lower() in weekdays:
            ret_entities['Wochentag'] = lemma_or_text

        # TODO: Belag: Referenz auf Pizza (head) und NN (xpos)
    return (ret_intent, ret_entities)

mask_opening_hours = df_opening_hours_test['nlu'].apply(nlu)
mask_orders = df_order_test['nlu'].apply(nlu)

mask_opening_hours
mask_orders

Eine vollständige NLU mit Regeln zu erstellen ist nicht ganz einfach. Dies zeigt das folgende mittels Github Copilot generierte Skript:

In [245]:
import json

WEEKDAYS = ['Montag', 'Dienstag', 'Mittwoch', 'Donnerstag', 'Freitag', 'Samstag', 'Sonntag']
SIZE_WORDS = {'klein', 'mittel', 'groß'}
CHEESE_LEMMAS = {'Käse'}
EXTRA_POS = {'extra', 'zusätzlich', 'doppelt', 'doppelter', 'mehr'}
NEGATION = {'ohne', 'kein'}
NON_TOPPING = {'bitte', 'gerne', 'größe', 'pizza', 'käse', 'cm'}


def nlu_generated(stanza_result: dict) -> tuple[str, dict]:
    ret_intent = None
    ret_entities = {}

    intent_lemmas_opening_hours = ['öffnen', 'geöffnet', 'offen', 'abholen', 'Öffnungszeit', 'Öffnungszeiten', 'auf', 'richtig', 'liefern']
    intent_lemmas_order = ['bestellen', 'Pizza', 'pizza']

    words = stanza_result[0]
    by_id = {w['id']: w for w in words}
    children = {}
    for w in words:
        children.setdefault(w.get('head', 0), []).append(w)

    def lemma(w):
        return w.get('lemma') or w['text']

    def is_cheese(w):
        return lemma(w) in CHEESE_LEMMAS or w['text'] in CHEESE_LEMMAS

    # ---------- Intent ----------
    weekday_found = None
    for w in words:
        if w['text'] in WEEKDAYS:
            weekday_found = w['text']

    if weekday_found and not any(lemma(w).lower() == 'bestellen' and w['text'].lower().startswith('bestell') and w.get('head') == 0 for w in words):
        # Wochentag + Öffnungs-Vokabular => Öffnungszeiten
        if any(lemma(w) in intent_lemmas_opening_hours or w['text'] in intent_lemmas_opening_hours for w in words):
            ret_intent = 'Oeffnungszeiten_Erfragen'

    if ret_intent is None:
        if any(lemma(w) in intent_lemmas_opening_hours or w['text'] in intent_lemmas_opening_hours for w in words) \
                and not any(lemma(w).lower() == 'pizza' and w.get('deprel') in ('obj', 'root') for w in words):
            ret_intent = 'Oeffnungszeiten_Erfragen'
        elif any(lemma(w) in intent_lemmas_order or w['text'].lower().startswith('pizza') for w in words) \
                or any(w['text'].lower().startswith('bestell') for w in words):
            ret_intent = 'Pizza_Bestellen'
        elif weekday_found:
            ret_intent = 'Oeffnungszeiten_Erfragen'

    # ---------- Entities ----------
    if ret_intent == 'Oeffnungszeiten_Erfragen':
        if weekday_found:
            ret_entities['Wochentag'] = weekday_found

    elif ret_intent == 'Pizza_Bestellen':
        # Größe
        for w in words:
            lem = lemma(w).lower()
            txt = w['text'].lower()
            # Zahlenangabe: "28 cm"
            if w['text'] == 'cm' or (w.get('upos') == 'NUM' and w['id'] < len(words)
                                      and by_id.get(w['id'] + 1, {}).get('text') == 'cm'):
                if w['text'] != 'cm':
                    ret_entities['Groesse'] = f"{w['text']} cm"
                continue
            if lem in SIZE_WORDS or txt in ('große', 'kleine', 'mittlere', 'grosse'):
                size = w['text']
                # "extra große": Modifier über den Dependency Tree
                mods = [c for c in children.get(w['id'], [])
                        if c['text'].lower() == 'extra' and c['id'] < w['id']]
                if mods:
                    size = f"extra {w['text']}"
                ret_entities['Groesse'] = size

        # Extra Käse (über Dependency Tree: Modifier / Präposition am Käse-Wort)
        for w in words:
            if is_cheese(w):
                related = children.get(w['id'], [])
                texts = {lemma(c).lower() for c in related} | {c['text'].lower() for c in related}
                if texts & NEGATION:
                    ret_entities['Extra_Kaese'] = 'nein'
                elif texts & EXTRA_POS or any(t.startswith(('zusätzlich', 'doppelt')) for t in texts):
                    ret_entities['Extra_Kaese'] = 'ja'
                else:
                    # Präposition (mit/ohne) hängt am Käse als 'case'
                    cases = {c['text'].lower() for c in related if c.get('deprel') == 'case'}
                    if cases & NEGATION:
                        ret_entities['Extra_Kaese'] = 'nein'
                    elif 'mit' in cases:
                        ret_entities['Extra_Kaese'] = 'ja'

        # Belag: Nomen/Eigennamen, die an "Pizza" hängen oder Pizza-Name sind
        skip = NON_TOPPING | {t.lower() for t in WEEKDAYS}
        topping = None
        pizza_ids = [w['id'] for w in words if lemma(w).lower().endswith('pizza') or w['text'].lower() == 'pizza']

        candidates = []
        for w in words:
            if w.get('upos') not in ('NOUN', 'PROPN', 'ADJ', 'X'):
                continue
            txt = w['text']
            if txt.lower() in skip or lemma(w).lower() in skip:
                continue
            if txt.lower() in ('große', 'kleine', 'mittlere', 'extra', 'extra große'):
                continue
            if w.get('upos') == 'ADJ' and txt.lower() not in ('vegetarische',):
                continue
            if w.get('upos') == 'NUM':
                continue
            if txt[0].isupper() or '-' in txt:
                candidates.append(w)

        # Bindestrich-Komposita wie "Thunfisch-Pizza"
        for w in words:
            if '-' in w['text'] and w['text'].lower().endswith('-pizza'):
                topping = w['text'].rsplit('-', 1)[0]
                break

        if topping is None:
            # bevorzugt Wörter, die direkt mit "Pizza" verbunden sind
            related = [c for c in candidates
                       if c.get('head') in pizza_ids or c['id'] in [by_id[p].get('head') for p in pizza_ids]]
            pool = related or candidates
            # Zusammengehörige Tokens (z.B. "Quattro Formaggi") per Dependency Tree zusammenfassen
            if pool:
                first = pool[0]
                group = [first] + [c for c in pool[1:]
                                   if c.get('head') == first['id'] or first.get('head') == c['id']
                                   or c.get('head') == first.get('head')]
                group = sorted({g['id']: g for g in group}.values(), key=lambda g: g['id'])
                topping = ' '.join(g['text'] for g in group)

        if topping:
            ret_entities['Belag'] = topping

    return (ret_intent, ret_entities)


def evaluate_nlu(df, expected_intent):
    results = df['nlu'].apply(nlu_generated)
    intent_ok = results.apply(lambda r: r[0] == expected_intent)
    entity_ok = [
        r[1] == json.loads(e) if isinstance(e, str) else False
        for r, e in zip(results, df['entities'])
    ]
    print(f"Intent korrekt:   {intent_ok.sum()} von {len(df)}")
    print(f"Entities korrekt: {sum(entity_ok)} von {len(df)}")


evaluate_nlu(df_opening_hours_test, 'Oeffnungszeiten_Erfragen')
evaluate_nlu(df_order_test, 'Pizza_Bestellen')

Intent korrekt:   80 von 80
Entities korrekt: 80 von 80
Intent korrekt:   66 von 80
Entities korrekt: 58 von 80
